## 依存ライブラリのインストール

データ生成に必要な`faker`ライブラリをインストールします。`faker`はリアルなダミーデータ（人名、企業名、メールアドレス等）を生成するために使用します。

## データ生成クラスの定義

以下のセルでは、データセット生成の中核となる2つのクラスを定義します。

- **`ChainOfThoughtGenerator`**: 各出力に付加する簡潔なCoT（思考連鎖）推論テキストを生成します。生成タスクと変換タスクでステップの内容が異なります。
- **`StructEvalDataGenerator`**: 17種類のスキーマ（研究論文、診療記録、API仕様など）に対応するフェイクデータを生成し、JSON / XML / YAML / TOML / CSV の5形式に変換する機能を提供します。

In [ ]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 26.0 MB/s eta 0:00:00


In [ ]:
from faker import Faker
import json
import yaml
import toml
import random
from typing import Dict, List, Tuple
from datetime import datetime, timedelta
import csv
from io import StringIO
from tqdm import tqdm

fake = Faker(['en_US'])

class ChainOfThoughtGenerator:
    """Generate concise Chain-of-Thought reasoning for structured data generation"""

    @staticmethod
    def generate_cot(format_type: str, schema_name: str, complexity: str, source_format: str = None) -> str:
        """Generate brief step-by-step reasoning (optimized for token limits)"""

        # Concise format requirements
        format_requirements = {
            "json": "proper braces/brackets, quoted keys, correct value types",
            "xml": "well-formed tags, proper nesting, escaped special chars",
            "yaml": "consistent indentation, colon-space syntax, hyphen lists",
            "toml": "key=value syntax, [sections], proper types",
            "csv": "header row, comma-separated, quoted fields with commas"
        }

        complexity_desc = {
            "simple": "3-5 fields, minimal nesting",
            "medium": "5-8 fields, 2-3 levels",
            "complex": "8-10 fields, 3-4 levels"
        }

        # Customize CoT for Conversion vs Generation
        if source_format:
            task_desc = f"Convert {schema_name.replace('_', ' ')} from {source_format.upper()} to {format_type.upper()}"
            step_5 = "5. Transform values to match the target format specifications."
        else:
            task_desc = f"Create {schema_name.replace('_', ' ')} in {format_type.upper()}"
            step_5 = "5. Populate fields with realistic example data based on the schema."

        # Build concise CoT
        cot = f"""Approach:
1. Task: {task_desc}
2. Complexity: {complexity} - {complexity_desc.get(complexity, 'standard')}
3. Format rules: {format_requirements.get(format_type, 'standard format')}
4. Structure: Organize data logically with appropriate nesting
{step_5}

Output:
"""
        return cot


class StructEvalDataGenerator:
    """Generate compact structured format data for MAX_SEQ_LEN=1536"""

    def __init__(self):
        self.fake = fake
        self.cot_generator = ChainOfThoughtGenerator()
        self.schema_generators = {
            'research_paper': self._gen_research_paper,
            'experiment_result': self._gen_experiment_result,
            'clinical_note': self._gen_clinical_note,
            'lab_result': self._gen_lab_result,
            'prescription': self._gen_prescription,
            'contract': self._gen_contract,
            'financial_report': self._gen_financial_report,
            'sales_analytics': self._gen_sales_analytics,
            'api_specification': self._gen_api_specification,
            'error_log': self._gen_error_log,
            'product_listing': self._gen_product_listing,
            'customer_review': self._gen_customer_review,
            'transaction_record': self._gen_transaction_record,
            'course_syllabus': self._gen_course_syllabus,
            'student_assessment': self._gen_student_assessment,
            'news_article': self._gen_news_article,
            'social_media_post': self._gen_social_media_post,
        }

    # ========== Compact Schema Generators ==========

    def _gen_research_paper(self) -> Dict:
        return {
            "paper_id": f"arXiv:{random.randint(2020, 2024)}.{random.randint(10000, 99999)}",
            "title": self.fake.catch_phrase(),
            "authors": [
                {
                    "name": self.fake.name(),
                    "affiliation": self.fake.company(),
                    "email": self.fake.email()
                } for _ in range(random.randint(2, 3))
            ],
            "abstract": self.fake.text(max_nb_chars=150),
            "keywords": [self.fake.word() for _ in range(random.randint(3, 5))],
            "publication": {
                "venue": random.choice(["Nature", "Science", "Cell"]),
                "year": random.randint(2020, 2024),
                "doi": f"10.{random.randint(1000, 9999)}/{self.fake.lexify(text='????')}"
            },
            "citations": random.randint(0, 500),
            "funding_source": random.choice([self.fake.company(), None])
        }

    def _gen_experiment_result(self) -> Dict:
        return {
            "experiment_id": f"EXP-{self.fake.random_int(100000, 999999)}",
            "title": f"Effect of {self.fake.word()} on {self.fake.word()}",
            "methodology": {
                "design": random.choice(["RCT", "observational", "cohort"]),
                "sample_size": random.randint(50, 1000),
                "duration_days": random.randint(30, 365)
            },
            "results": [
                {
                    "metric": random.choice(["efficacy", "safety"]),
                    "baseline": round(random.uniform(10, 100), 2),
                    "endpoint": round(random.uniform(10, 100), 2),
                    "p_value": round(random.uniform(0.001, 0.5), 4)
                } for _ in range(random.randint(2, 4))
            ],
            "conclusion": random.choice(["significant", "non-significant"])
        }

    def _gen_clinical_note(self) -> Dict:
        return {
            "note_id": f"CN-{self.fake.random_int(1000000, 9999999)}",
            "patient": {
                "mrn": f"MRN-{self.fake.random_int(100000, 999999)}",
                "age": random.randint(18, 90),
                "gender": random.choice(["M", "F"])
            },
            "encounter_date": self.fake.date_this_year().isoformat(),
            "chief_complaint": random.choice(["chest pain", "fever", "headache"]),
            "vitals": {
                "bp": f"{random.randint(90, 180)}/{random.randint(60, 110)}",
                "hr": random.randint(60, 120),
                "temp_f": round(random.uniform(97.0, 103.0), 1)
            },
            "assessment": [
                {
                    "problem": random.choice(["hypertension", "diabetes"]),
                    "icd10": f"{random.choice('IJK')}{random.randint(10, 99)}"
                } for _ in range(random.randint(1, 3))
            ]
        }

    def _gen_lab_result(self) -> Dict:
        return {
            "result_id": f"LAB-{self.fake.random_int(1000000, 9999999)}",
            "patient_mrn": f"MRN-{self.fake.random_int(100000, 999999)}",
            "order_date": self.fake.date_this_month().isoformat(),
            "panel": random.choice(["CMP", "CBC", "Lipid"]),
            "results": [
                {
                    "test": random.choice(["Glucose", "Hemoglobin", "WBC"]),
                    "value": round(random.uniform(50, 200), 1),
                    "unit": random.choice(["mg/dL", "g/dL"]),
                    "flag": random.choice(["normal", "high", "low"])
                } for _ in range(random.randint(4, 7))
            ]
        }

    def _gen_prescription(self) -> Dict:
        return {
            "rx_id": f"RX-{self.fake.random_int(1000000, 9999999)}",
            "patient_mrn": f"MRN-{self.fake.random_int(100000, 999999)}",
            "prescriber": self.fake.name(),
            "date": self.fake.date_this_month().isoformat(),
            "medications": [
                {
                    "name": random.choice(["Lisinopril", "Metformin", "Atorvastatin"]),
                    "dosage": f"{random.choice([5, 10, 20])}mg",
                    "frequency": random.choice(["daily", "BID", "TID"]),
                    "quantity": random.choice([30, 60, 90])
                } for _ in range(random.randint(1, 3))
            ]
        }

    def _gen_contract(self) -> Dict:
        return {
            "contract_id": f"CTR-{self.fake.random_int(100000, 999999)}",
            "parties": [
                {"name": self.fake.company(), "role": "party_a"},
                {"name": self.fake.company(), "role": "party_b"}
            ],
            "effective_date": self.fake.date_this_year().isoformat(),
            "term_months": random.randint(12, 60),
            "value": random.randint(10000, 1000000),
            "status": random.choice(["active", "pending", "executed"])
        }

    def _gen_financial_report(self) -> Dict:
        return {
            "company": self.fake.company(),
            "ticker": self.fake.lexify(text='????').upper(),
            "quarter": f"Q{random.randint(1, 4)} {random.randint(2020, 2024)}",
            "revenue": random.randint(1000000, 100000000),
            "expenses": random.randint(500000, 80000000),
            "net_income": random.randint(50000, 20000000),
            "eps": round(random.uniform(0.5, 10.0), 2),
            "metrics": {
                "profit_margin": round(random.uniform(5.0, 30.0), 2),
                "roe": round(random.uniform(5.0, 25.0), 2)
            }
        }

    def _gen_sales_analytics(self) -> Dict:
        return {
            "period": f"{self.fake.month_name()} {random.randint(2020, 2024)}",
            "total_revenue": random.randint(500000, 10000000),
            "units_sold": random.randint(1000, 100000),
            "channels": [
                {
                    "name": ch,
                    "revenue": random.randint(100000, 5000000),
                    "growth": round(random.uniform(-10.0, 50.0), 2)
                } for ch in random.sample(["online", "retail", "wholesale"], 2)
            ],
            "top_products": [
                {"sku": f"SKU-{random.randint(1000, 9999)}", "sales": random.randint(10000, 500000)}
                for _ in range(3)
            ]
        }

    def _gen_api_specification(self) -> Dict:
        return {
            "openapi": "3.0.0",
            "info": {
                "title": self.fake.catch_phrase() + " API",
                "version": "1.0.0"
            }, # Fixed: Added missing closing curly brace '}'
            "paths": {
                f"/{self.fake.word()}": {
                    "get": {
                        "summary": self.fake.sentence()[:50],
                        "parameters": [
                            {"name": f"param{i}", "in": "query", "type": "string"}
                            for i in range(random.randint(1, 3))
                        ],
                        "responses": {
                            "200": {"description": "Success"},
                            "400": {"description": "Bad Request"}
                        }
                    }
                }
            }
        }

    def _gen_error_log(self) -> Dict:
        return {
            "error_id": self.fake.uuid4(),
            "timestamp": datetime.now().isoformat(),
            "level": random.choice(["ERROR", "CRITICAL", "WARNING"]),
            "service": self.fake.word() + "Service",
            "message": self.fake.sentence()[:80],
            "stack_trace": [
                f"at {self.fake.word()}.py:{random.randint(10, 500)}"
                for _ in range(random.randint(2, 4))
            ],
            "context": {
                "user_id": self.fake.uuid4(),
                "request_id": self.fake.uuid4()
            }
        }

    def _gen_product_listing(self) -> Dict:
        stock_count = random.randint(0, 1000)
        return {
            "product_id": f"PROD-{self.fake.random_int(100000, 999999)}",
            "name": self.fake.catch_phrase(),
            "category": random.choice(["electronics", "clothing", "books"]),
            "price": round(random.uniform(10, 1000), 2),
            "currency": "USD",
            "stock": stock_count,
            "is_available": stock_count > 0,
            "rating": round(random.uniform(1.0, 5.0), 2),
            "reviews": random.randint(0, 1000),
            "specs": {
                "weight_kg": round(random.uniform(0.1, 50), 2),
                "dimensions": f"{random.randint(10, 100)}x{random.randint(10, 100)}x{random.randint(10, 100)}cm"
            }
        }

    def _gen_customer_review(self) -> Dict:
        return {
            "review_id": f"REV-{self.fake.random_int(1000000, 9999999)}",
            "product_id": f"PROD-{self.fake.random_int(100000, 999999)}",
            "user_id": self.fake.uuid4(),
            "rating": random.randint(1, 5),
            "title": self.fake.sentence()[:50],
            "content": self.fake.text(max_nb_chars=200),
            "date": self.fake.date_this_year().isoformat(),
            "verified_purchase": random.choice([True, False]),
            "helpful_votes": random.randint(0, 100)
        }

    def _gen_transaction_record(self) -> Dict:
        return {
            "txn_id": f"TXN-{self.fake.random_int(10000000, 99999999)}",
            "customer_id": self.fake.uuid4(),
            "timestamp": datetime.now().isoformat(),
            "items": [
                {
                    "sku": f"SKU-{random.randint(10000, 99999)}",
                    "qty": random.randint(1, 5),
                    "price": round(random.uniform(10, 500), 2)
                } for _ in range(random.randint(1, 4))
            ],
            "subtotal": round(random.uniform(50, 2000), 2),
            "tax": round(random.uniform(5, 180), 2),
            "total": round(random.uniform(55, 2200), 2),
            "payment_method": random.choice(["credit_card", "paypal", "apple_pay"])
        }

    def _gen_course_syllabus(self) -> Dict:
        return {
            "course_id": f"CRS-{random.randint(1000, 9999)}",
            "code": f"{self.fake.lexify(text='???').upper()}{random.randint(100, 499)}",
            "title": self.fake.catch_phrase() + " 101",
            "instructor": self.fake.name(),
            "credits": random.choice([3, 4]),
            "term": f"{random.choice(['Fall', 'Spring'])} {random.randint(2020, 2024)}",
            "schedule": [
                {"week": i, "topic": self.fake.word()}
                for i in range(1, random.randint(8, 12))
            ],
            "grading": {
                "homework": 30,
                "midterm": 30,
                "final": 40
            }
        }

    def _gen_student_assessment(self) -> Dict:
        return {
            "assessment_id": f"ASMT-{self.fake.random_int(100000, 999999)}",
            "student_id": f"STU-{random.randint(100000, 999999)}",
            "course_id": f"CRS-{random.randint(1000, 9999)}",
            "type": random.choice(["exam", "quiz", "assignment"]),
            "date": self.fake.date_this_year().isoformat(),
            "scores": [
                {
                    "question": i,
                    "points_earned": random.randint(0, 10),
                    "points_possible": 10
                } for i in range(1, random.randint(8, 15))
            ],
            "total_score": random.randint(50, 100),
            "percentage": round(random.uniform(50, 100), 2),
            "grade": random.choice(["A", "B", "C"])
        }

    def _gen_news_article(self) -> Dict:
        return {
            "article_id": f"ART-{self.fake.random_int(1000000, 9999999)}",
            "headline": self.fake.catch_phrase(),
            "author": self.fake.name(),
            "published": self.fake.date_time_this_year().isoformat(),
            "section": random.choice(["Politics", "Business", "Tech", "Sports"]),
            "content": self.fake.text(max_nb_chars=300),
            "tags": [self.fake.word() for _ in range(random.randint(3, 5))],
            "views": random.randint(100, 100000),
            "shares": random.randint(10, 5000)
        }

    def _gen_social_media_post(self) -> Dict:
        return {
            "post_id": f"POST-{self.fake.random_int(10000000, 99999999)}",
            "platform": random.choice(["twitter", "facebook", "instagram"]),
            "user_id": self.fake.uuid4(),
            "username": self.fake.user_name(),
            "content": self.fake.text(max_nb_chars=200),
            "timestamp": datetime.now().isoformat(),
            "hashtags": [f"#{self.fake.word()}" for _ in range(random.randint(2, 4))],
            "engagement": {
                "likes": random.randint(0, 10000),
                "shares": random.randint(0, 1000),
                "comments": random.randint(0, 500)
            }
        }

    # ========== Format Converters ==========

    def to_json(self, data: Dict) -> str:
        return json.dumps(data, ensure_ascii=False, indent=2)

    def to_xml(self, data: Dict, root: str = "root") -> str:
        def dict_to_xml(d, tag):
            parts = [f"<{tag}>"]
            for k, v in d.items():
                clean_key = k.replace(' ', '_').replace('-', '_')
                if isinstance(v, dict):
                    parts.append(dict_to_xml(v, clean_key))
                elif isinstance(v, list):
                    for item in v:
                        if isinstance(item, dict):
                            parts.append(dict_to_xml(item, clean_key))
                        else:
                            parts.append(f"<{clean_key}>{self._escape_xml(str(item))}</{clean_key}>")
                elif v is None:
                    parts.append(f"<{clean_key} />")
                else:
                    parts.append(f"<{clean_key}>{self._escape_xml(str(v))}</{clean_key}>")
            parts.append(f"</{tag}>")
            return "\n".join(parts)

        return f'<?xml version="1.0" encoding="UTF-8"?>\n{dict_to_xml(data, root)}'

    def _escape_xml(self, text: str) -> str:
        return (text.replace("&", "&amp;")
                   .replace("<", "&lt;")
                   .replace(">", "&gt;")
                   .replace('"', "&quot;"))

    def to_yaml(self, data: Dict) -> str:
        return yaml.dump(data, allow_unicode=True, default_flow_style=False, sort_keys=False)

    def to_toml(self, data: Dict) -> str:
        try:
            return toml.dumps(data)
        except Exception:
            simplified = self._simplify_for_toml(data)
            return toml.dumps(simplified)

    def _simplify_for_toml(self, data: Dict, prefix: str = '') -> Dict:
        result = {}
        for key, value in data.items():
            new_key = f"{prefix}.{key}" if prefix else key
            if isinstance(value, dict):
                if all(not isinstance(v, (dict, list)) for v in value.values()):
                    result[new_key] = value
                else:
                    result.update(self._simplify_for_toml(value, new_key))
            elif isinstance(value, list):
                if value and isinstance(value[0], dict):
                    result[new_key] = json.dumps(value[:5])
                else:
                    result[new_key] = value[:10]
            else:
                result[new_key] = value
        return result

    def to_csv(self, data: Dict) -> str:
        def flatten_dict(d, parent_key='', sep='_'):
            items = []
            for k, v in d.items():
                new_key = f"{parent_key}{sep}{k}" if parent_key else k
                if isinstance(v, dict):
                    items.extend(flatten_dict(v, new_key, sep=sep).items())
                elif isinstance(v, list):
                    items.append((new_key, json.dumps(v[:5], ensure_ascii=False)))
                else:
                    items.append((new_key, v))
            return dict(items)

        rows = [flatten_dict(data)]
        schema_name = self._get_schema_name(data)
        if schema_name in self.schema_generators:
            for _ in range(random.randint(2, 4)):
                extra_data = self.schema_generators[schema_name]()
                rows.append(flatten_dict(extra_data))

        output = StringIO()
        if rows:
            all_keys = set()
            for row in rows:
                all_keys.update(row.keys())
            fieldnames = sorted(all_keys)

            writer = csv.DictWriter(output, fieldnames=fieldnames)
            writer.writeheader()
            for row in rows:
                filled_row = {k: str(row.get(k, ''))[:100] for k in fieldnames}
                writer.writerow(filled_row)

        return output.getvalue()

    def _get_schema_name(self, data: Dict) -> str:
        key_mappings = {
            'paper_id': 'research_paper',
            'experiment_id': 'experiment_result',
            'note_id': 'clinical_note',
            'result_id': 'lab_result',
            'rx_id': 'prescription',
            'contract_id': 'contract',
            'company': 'financial_report',
            'product_id': 'product_listing',
            'review_id': 'customer_review',
            'txn_id': 'transaction_record',
            'course_id': 'course_syllabus',
            'assessment_id': 'student_assessment',
            'article_id': 'news_article',
            'post_id': 'social_media_post'
        }

        for key, schema in key_mappings.items():
            if key in data:
                return schema

        return list(self.schema_generators.keys())[0]

    def generate_base_data(self, complexity: str = "medium") -> Tuple[Dict, str]:
        schema_name = random.choice(list(self.schema_generators.keys()))
        data = self.schema_generators[schema_name]()

        if complexity == "simple":
            keys = list(data.keys())
            keep_keys = random.sample(keys, min(len(keys), random.randint(3, 5)))
            data = {k: data[k] for k in keep_keys}
        # complex uses full schema

        return data, schema_name

In [ ]:
def estimate_tokens(text: str) -> int:
    """Rough token estimation (1 token \u2245 4 chars for English)"""
    return len(text) // 4

def generate_dataset(num_base_examples: int = 500, output_file: str = "struct_eval_train_cot.jsonl", max_tokens: int = 1200):
    """
    Generate dataset with Chain-of-Thought reasoning, including format conversions.
    """

    generator = StructEvalDataGenerator()
    formats = ["json", "xml", "yaml", "toml", "csv"]
    complexities = ["simple", "medium", "complex"]

    all_examples = []
    skipped = 0

    print(f"Generating {num_base_examples} base examples (Generation + Conversion)...")

    for _ in tqdm(range(num_base_examples), desc="Base examples"):
        complexity = random.choices(complexities, weights=[0.2, 0.4, 0.4])[0]
        base_data, schema_name = generator.generate_base_data(complexity)

        format_funcs = {
            "json": generator.to_json,
            "xml": lambda d: generator.to_xml(d, schema_name),
            "yaml": generator.to_yaml,
            "toml": generator.to_toml,
            "csv": generator.to_csv
        }

        # 1. Text -> Format (Generation)
        for format_type in formats:
            try:
                output = format_funcs[format_type](base_data)
                cot = generator.cot_generator.generate_cot(format_type, schema_name, complexity)
                assistant_response = f"{cot}{output}"

                if estimate_tokens(assistant_response) > max_tokens:
                    skipped += 1
                    continue

                prompts = [
                    f"Generate {schema_name.replace('_', ' ')} data in {format_type.upper()} format.",
                    f"Create a {format_type.upper()} representation of {schema_name.replace('_', ' ')}.",
                    f"Output {schema_name.replace('_', ' ')} as {format_type.upper()}.",
                    f"Produce a {format_type.upper()} document for a {schema_name.replace('_', ' ')}."
                ]

                all_examples.append({
                    "format": format_type,
                    "complexity": complexity,
                    "schema": schema_name,
                    "type": "generation",
                    "prompt": random.choice(prompts),
                    "output": assistant_response,
                    "estimated_tokens": estimate_tokens(assistant_response)
                })
            except Exception as e:
                continue

        # 2. Format -> Format (Conversion)
        all_possible_conversion_pairs = [
            (f1, f2) for f1 in formats for f2 in formats if f1 != f2
        ]

        # Prioritize problematic pairs and ensure diversity
        conversion_tasks_for_batch = []
        # Ensure json<->xml are always present
        conversion_tasks_for_batch.append(('json', 'xml'))
        conversion_tasks_for_batch.append(('xml', 'json'))

        # Add more diverse pairs, ensuring at least 3 unique tasks total per base example
        # Filter out already added pairs
        remaining_pairs = [p for p in all_possible_conversion_pairs if p not in conversion_tasks_for_batch]
        # Add up to 1 more unique random pair if possible to reach 3 total
        if len(remaining_pairs) > 0:
            conversion_tasks_for_batch.extend(random.sample(remaining_pairs, min(1, len(remaining_pairs))))

        # Shuffle to randomize the order if multiple options are chosen, but preserve specific pairs if only 2
        random.shuffle(conversion_tasks_for_batch)

        for src_fmt, tgt_fmt in conversion_tasks_for_batch:
            try:
                src_data = format_funcs[src_fmt](base_data)
                tgt_data = format_funcs[tgt_fmt](base_data)

                cot = generator.cot_generator.generate_cot(tgt_fmt, schema_name, complexity, source_format=src_fmt)
                assistant_response = f"{cot}{tgt_data}"

                # Input prompt length + output length check could be strict, but mainly checking output for now
                if estimate_tokens(assistant_response) > max_tokens:
                    skipped += 1
                    continue

                prompts = [
                    f"Convert the following {src_fmt.upper()} to {tgt_fmt.upper()}:\n{src_data}",
                    f"Transform this {src_fmt.upper()} data into {tgt_fmt.upper()} format:\n{src_data}",
                    f"Reformat the data below from {src_fmt.upper()} to {tgt_fmt.upper()}:\n{src_data}"
                ]

                all_examples.append({
                    "format": tgt_fmt,
                    "complexity": complexity,
                    "schema": schema_name,
                    "type": "conversion",
                    "prompt": random.choice(prompts),
                    "output": assistant_response,
                    "estimated_tokens": estimate_tokens(assistant_response)
                })
            except Exception:
                continue

    random.shuffle(all_examples)

    print(f"\nSaving {len(all_examples)} examples (skipped {skipped} for length)...")
    formatted_examples = []

    for ex in all_examples:
        messages = [
            {
                "role": "system",
                "content": f"You are an expert in {ex['format'].upper()} format. Think step-by-step, then generate syntactically perfect {ex['format'].upper()}."
            },
            {
                "role": "user",
                "content": ex["prompt"]
            },
            {
                "role": "assistant",
                "content": ex["output"]
            }
        ]

        formatted_examples.append({
            "messages": messages,
            "metadata": {
                "format": ex["format"],
                "complexity": ex["complexity"],
                "schema": ex["schema"],
                "type": ex["type"],
                "estimated_tokens": ex["estimated_tokens"]
            }
        })

    with open(output_file, 'w', encoding='utf-8') as f:
        for example in formatted_examples:
            f.write(json.dumps(example, ensure_ascii=False) + '\n')

    # Statistics
    print(f"\n{'='*60}")
    print("Dataset Statistics")
    print(f"{'='*60}")
    print(f"Total examples: {len(formatted_examples)}")
    print(f"Output file: {output_file}")

    from collections import Counter
    types = [ex['metadata']['type'] for ex in formatted_examples]
    formats = [ex['metadata']['format'] for ex in formatted_examples]

    print("\nTask Type distribution:")
    for t, count in Counter(types).items():
        print(f"  {t}: {count}")

    print("\nFormat distribution:")
    for fmt, count in Counter(formats).items():
        print(f"  {fmt.upper()}: {count}")

    print(f"\n\u2713 Dataset generation completed!")

### データセット生成の実行方法

`generate_dataset` 関数には、生成する基本例の数と出力ファイル名を指定できます。

*   `num_base_examples`: 生成する基本例の数（デフォルトは500）。各基本例は5つの異なる形式に変換されるため、合計で `num_base_examples * 5` の例が生成されます。
*   `output_file`: 生成されたデータセットを保存するファイル名（デフォルトは `struct_eval_train.jsonl`）。

デフォルト値を使用するには、引数なしで関数を呼び出すだけです。カスタム値を指定するには、引数を渡します。

In [ ]:
DATA_FILE = 'structured_data_with_cot_dataset_512_v2.jsonl'

In [ ]:
# デフォルト値（num_base_examples=500, output_file='structured_data_with_cot_dataset.jsonl'）でデータセットを生成する
# generate_dataset()

# 例：200個の基本例を生成し、'my_custom_dataset.jsonl' に保存する
# generate_dataset(num_base_examples=200, output_file='my_custom_dataset.jsonl')

generate_dataset(num_base_examples=500, output_file=DATA_FILE, max_tokens=512)

Generating 500 base examples (Generation + Conversion)...


Base examples: 100%|██████████| 500/500 [00:02<00:00, 191.15it/s]



Saving 3933 examples (skipped 67 for length)...

Dataset Statistics
Total examples: 3933
Output file: structured_data_with_cot_dataset_512_v2.jsonl

Task Type distribution:
  conversion: 1485
  generation: 2448

Format distribution:
  TOML: 611
  CSV: 547
  XML: 1076
  JSON: 1075
  YAML: 624

✓ Dataset generation completed!


上記のセルを実行すると、指定された数のデータが生成され、進捗状況が表示された後、結果がファイルに保存されます。

データセットの生成が完了したら、`structured_data_with_cot_dataset_512_v2.jsonl`（または指定したファイル名）の内容を確認できます。

## データセットの読み込み

生成されたJSONLファイル（`structured_data_with_cot_dataset_512_v2.jsonl`）をHugging Faceの`Dataset`オブジェクトとして読み込みます。

`datasets`ライブラリの`load_dataset`関数を使用して、JSONLファイルを読み込みます。`data_files`にファイルパスを、`split`に`'train'`を指定します。

In [ ]:
from datasets import load_dataset

# Load the JSONL file into a Hugging Face Dataset object
dataset = load_dataset('json', data_files=DATA_FILE, split='train')

# Display the loaded dataset to confirm
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages', 'metadata'],
    num_rows: 3933
})


## Hugging Face Hubへのアップロード

読み込んだデータセットをHugging Face Hubにプッシュします。事前にリポジトリ名とアクセストークンの設定が必要です。

In [ ]:
import os
from google.colab import userdata

# HF_TOKENを環境変数またはColabの秘密鍵から取得
# Colabの左側のパネルにある「秘密鍵（🔑）」アイコンをクリックし、
# 「HF_TOKEN」という名前でトークンを設定してください。
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    HF_TOKEN = userdata.get('HFH_FG') # 秘密鍵サービスから取得

if not HF_TOKEN:
    print("Warning: HF_TOKEN not found. Hugging Face login might fail.")

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=True)

In [ ]:
readme_lines = [
    "# structured_data_with_cot_dataset",
    "",
    "このデータセットは、様々な形式（JSON、XML、YAML、TOML、CSV）の構造化データと、それぞれに対応する簡潔な思考連鎖（Chain-of-Thought, CoT）推論を含む多様な例を提供します。このデータセットの目的は、自然言語プロンプトに基づいて構造化データを生成できるモデルに対し、CoTを活用してより良い推論と出力品質を達成するための高品質な学習例を提供することです。",
    "",
    "## データセットの概要",
    "",
    "データセットの各エントリには、以下のものが含まれます。",
    "- **`messages`**: OpenAIチャット形式に従ったメッセージ辞書のリストで、以下で構成されます。",
    "  - `system`メッセージ: 特定のデータ形式の専門家としてのコンテキストを設定します。",
    "  - `user`メッセージ: 特定のスキーマと形式で構造化データの生成を要求するプロンプトです。多様なプロンプトテンプレートが使用されています。",
    "  - `assistant`メッセージ: 生成された構造化データに続く思考連鎖（CoT）推論が含まれます。",
    "- **`metadata`**: 例に関する追加情報で、以下が含まれます。",
    "  - `format`: 構造化データの形式（例: 'json'、'xml'、'yaml'、'toml'、'csv'）。",
    "  - `complexity`: 生成されたスキーマの複雑度（'simple'、'medium'、'complex'）。より複雑な例が多く含まれるように調整されています。",
    "  - `schema`: 表現される実世界のスキーマの種類（例: 'research_paper'、'clinical_note'、'api_specification'）。",
    "  - `estimated_tokens`: アシスタントの応答のトークン数の推定値。",
    "",
    "## サポートされるデータ形式",
    "",
    "- **JSON** (JavaScript Object Notation)",
    "- **XML** (Extensible Markup Language)",
    "- **YAML** (YAML Ain't Markup Language)",
    "- **TOML** (Tom's Obvious, Minimal Language) - 複雑なネストは簡素化される場合があります。",
    "- **CSV** (Comma Separated Values) - 複数の行が生成され、フラット化された形式で表現されます。",
    "",
    "## サポートされるスキーマ",
    "",
    "このデータセットは、以下を含む幅広いコンパクトなスキーマをカバーしています。",
    "- 研究論文 (例: `funding_source`のような条件付きフィールドを含む)",
    "- 実験結果",
    "- 診療記録",
    "- 検査結果",
    "- 処方箋",
    "- 契約書",
    "- 財務報告書",
    "- 販売分析",
    "- API仕様",
    "- エラーログ",
    "- 製品リスト (例: `is_available`のような条件付きフィールドを含む)",
    "- 顧客レビュー",
    "- 取引記録",
    "- シラバス",
    "- 学生評価",
    "- ニュース記事",
    "- ソーシャルメディア投稿",
    "",
    "## 思考連鎖（Chain-of-Thought, CoT）推論",
    "",
    "各`assistant`メッセージは、構造化データを生成するために取られたステップを概説する短いCoT推論ブロックから始まります。これにより、モデルは構造化データ生成に関わる論理と制約を理解し、正確で文脈に関連する出力を生成する能力を向上させることができます。CoTには、タスク、複雑度、形式ルール、構造、フィールドへのデータ入力に関する一般的なステップが含まれます。",
    "",
    "## データセットの読み込み方法",
    "",
    "このデータセットは`JSONL`（JSON Lines）形式で提供されています。Hugging Faceの`datasets`ライブラリを使用して簡単にロードできます。",
    "",
    "```python",
    "from datasets import load_dataset",
    "",
    "# ローカルのJSONLファイルからデータセットをロード",
    "dataset = load_dataset('json', data_files='structured_data_with_cot_dataset.jsonl', split='train')",
    "",
    "# 例にアクセス",
    "print(dataset[0])",
    "",
    "# このデータセットをHugging Face Hubにプッシュすることもできます",
    "# from huggingface_hub import notebook_login",
    "# notebook_login() # ログインしていない場合",
    "# dataset.push_to_hub(\"your-username/structured_data_with_cot_dataset\")",
    "```",
    "",
    "## 生成方法",
    "",
    "このデータセットは、`Faker`ライブラリを活用したPythonスクリプトを使用して、様々なコンパクトなスキーマに対して現実的な構造化データを生成しました。生成プロセスには、スキーマ、形式、複雑度をランダムに選択し、思考連鎖推論を埋め込むロジックが含まれており、大規模言語モデルのトレーニングに適した多様性と品質を保証します。特に、より多様なプロンプトテンプレートが使用され、データセットの複雑度の分布が調整され（より多くの「medium」および「complex」な例）、いくつかのスキーマジェネレーターには条件付きフィールド（例：研究論文の資金源、製品リストの在庫状況）や、CSVの動的な複数行生成ロジックが追加されています。TOML変換は、TOMLの仕様に合わせてネストされた構造を簡素化する場合があります。",
    ""
]

readme_content = "\n".join(readme_lines)

with open('README.md', 'w', encoding='utf-8') as f:
    f.write(readme_content)

print("README.md file created successfully.")

README.md file created successfully.


In [11]:
import os
from huggingface_hub import HfApi

repo_name = "u-10bei/structured_data_with_cot_dataset_512_v2"

# Define YAML front matter for the dataset card
yaml_front_matter = """
---
license: mit
tags:
- json
- xml
- yaml
- toml
- csv
- structured_data
- cot
- chain-of-thought
- data_generation
pretty_name: Structured Data with CoT (512 tokens)
---
"""

# Re-load the original readme content (assuming it's still in memory or regenerated)
# If readme_content is not in memory, we would need to read it from file.
# For this execution, I'll use the 'readme_content' variable from the previous state.
# In a real scenario, you might read from 'README.md' if the variable was cleared.

# The variable `readme_lines` was defined in a previous cell.
readme_content = "\n".join(readme_lines)

# Combine YAML front matter with the existing README content
full_readme_content = yaml_front_matter + readme_content

# Write the updated content to README.md
with open('README.md', 'w', encoding='utf-8') as f:
    f.write(full_readme_content)

print("Updated README.md with YAML metadata successfully.")

# Upload the updated README.md file to the same repository
hf_api = HfApi()
hf_api.upload_file(
    path_or_fileobj="README.md",
    path_in_repo="README.md",
    repo_id=repo_name,
    repo_type="dataset",
    token=HF_TOKEN
)
print(f"Updated README.md successfully uploaded to https://huggingface.co/datasets/{repo_name}/blob/main/README.md")

Updated README.md with YAML metadata successfully.
Updated README.md successfully uploaded to https://huggingface.co/datasets/u-10bei/structured_data_with_cot_dataset_512_v2/blob/main/README.md
